# Projet de synthèse - Développement humain en Afrique
## Étape 1 : Nettoyage & préparation des données (Python / Pandas)

**AnalystLab Africa - Semaine 8 (Capstone)**

**Source des données :** World Development Indicators (WDI), Banque Mondiale.

**Objectif du notebook :** transformer le fichier brut `WDICSV.csv` (~397 000 lignes, format large, 2 204 indicateurs, 259 zones) en un jeu de données **propre, structuré et prêt pour Power BI** : une ligne par *pays × année*, avec nos 12 indicateurs de développement humain en colonnes, enrichi de la région, du groupe de revenu et de la sous-région africaine.

**Résultat :** `Afrique_clean.csv` (~54 pays × 24 ans ≈ 1 296 lignes).

---
## 1. Configuration : imports et paramètres du projet

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

# Dossiers
DATA = Path('data')
OUT  = Path('.')  # le CSV propre sera écrit dans le dossier WEEK 8

# --- Les 12 indicateurs retenus (code WDI -> nom lisible) ---
INDICATEURS = {
    'SP.DYN.LE00.IN'   : 'LifeExpectancy',      # Espérance de vie à la naissance (années)
    'SH.DYN.MORT'      : 'U5Mortality',         # Mortalité des moins de 5 ans (/1000)
    'SH.STA.MMRT'      : 'MaternalMortality',   # Mortalité maternelle (/100 000 naissances)
    'SP.DYN.TFRT.IN'   : 'FertilityRate',       # Indice de fécondité (enfants/femme)
    'SE.PRM.ENRR'      : 'PrimaryEnrollment',   # Taux de scolarisation primaire brut (%)
    'SE.ADT.LITR.ZS'   : 'AdultLiteracy',       # Alphabétisation des adultes (%)
    'EG.ELC.ACCS.ZS'   : 'ElectricityAccess',   # Accès à l'électricité (% population)
    'SH.H2O.BASW.ZS'   : 'WaterAccess',         # Accès à l'eau potable de base (%)
    'SH.STA.BASS.ZS'   : 'SanitationAccess',    # Accès à l'assainissement de base (%)
    'NY.GDP.PCAP.CD'   : 'GDPperCapita',        # PIB par habitant (US$ courants)
    'SP.POP.TOTL'      : 'Population',           # Population totale
    'IT.NET.USER.ZS'   : 'InternetUsers',       # Utilisateurs d'Internet (% population)
}

# --- Les 54 pays africains (codes ISO-3) ---
PAYS_AFRIQUE = [
    'AGO','BDI','BEN','BFA','BWA','CAF','CIV','CMR','COD','COG','COM','CPV','DJI','DZA',
    'EGY','ERI','ETH','GAB','GHA','GIN','GMB','GNB','GNQ','KEN','LBR','LBY','LSO','MAR',
    'MDG','MLI','MOZ','MRT','MUS','MWI','NAM','NER','NGA','RWA','SDN','SEN','SLE','SOM',
    'SSD','STP','SWZ','SYC','TCD','TGO','TUN','TZA','UGA','ZAF','ZMB','ZWE'
]

# --- Sous-régions africaines (valeur ajoutée pour les filtres du dashboard) ---
SOUS_REGION = {
    **{c: 'Afrique du Nord'  for c in ['DZA','EGY','LBY','MAR','TUN','SDN']},
    **{c: "Afrique de l'Ouest" for c in ['BEN','BFA','CIV','CPV','GHA','GIN','GMB','GNB','LBR','MLI','MRT','NER','NGA','SEN','SLE','TGO']},
    **{c: "Afrique de l'Est" for c in ['BDI','COM','DJI','ERI','ETH','KEN','MDG','MUS','MWI','MOZ','RWA','SYC','SOM','SSD','TZA','UGA','ZMB','ZWE']},
    **{c: 'Afrique centrale' for c in ['AGO','CAF','CMR','COD','COG','GAB','GNQ','STP','TCD']},
    **{c: 'Afrique australe' for c in ['BWA','LSO','NAM','SWZ','ZAF']},
}

ANNEE_MIN = 2000
ANNEE_MAX = 2023
print(f"{len(INDICATEURS)} indicateurs | {len(PAYS_AFRIQUE)} pays | période {ANNEE_MIN}-{ANNEE_MAX}")

12 indicateurs | 54 pays | période 2000-2023


---
## 2. Chargement & compréhension du jeu de données brut

In [2]:
df_raw = pd.read_csv(DATA / 'WDICSV.csv')
print('Dimensions du fichier brut :', df_raw.shape)
print('Colonnes (début) :', list(df_raw.columns[:6]), '...', list(df_raw.columns[-3:]))
df_raw.head(3)

Dimensions du fichier brut : (396970, 70)
Colonnes (début) : ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961'] ... ['2023', '2024', '2025']


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,...,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,16.123198,16.599734,17.061049,17.634150,18.145833,18.685118,19.205632,19.742772,20.332679,20.862800,21.419621,21.996456,22.541440,NaN,NaN
1,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.RU.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.028196,6.309218,6.643625,6.932310,7.259936,7.606712,7.926604,8.309896,8.704591,9.106640,9.480804,9.903209,10.288154,NaN,NaN
2,Africa Eastern and Southern,AFE,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.UR.ZS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,37.252570,37.635987,37.969321,38.340232,38.741988,39.052626,39.321068,39.649534,39.968299,40.354628,40.723805,41.026351,41.289974,NaN,NaN


In [3]:
# Nombre d'indicateurs et de zones présents dans le fichier
print('Indicateurs distincts :', df_raw['Indicator Code'].nunique())
print('Pays / zones distincts :', df_raw['Country Code'].nunique())
print('Le format est LARGE : une colonne par année (1960 -> 2024), à dépivoter.')

Indicateurs distincts : 1498
Pays / zones distincts : 265
Le format est LARGE : une colonne par année (1960 -> 2024), à dépivoter.


**Constat :** le fichier est en *format large* et couvre le monde entier. Nous devons (a) filtrer sur l'Afrique et nos indicateurs, (b) dépivoter les années, (c) repivoter les indicateurs en colonnes.

---
## 3. Filtrage : 54 pays africains + 12 indicateurs

Première réduction massive : de ~397 000 lignes à ~648 lignes (54 × 12).

In [4]:
df = df_raw[
    df_raw['Country Code'].isin(PAYS_AFRIQUE) &
    df_raw['Indicator Code'].isin(INDICATEURS.keys())
].copy()
print('Après filtrage :', df.shape)

# Contrôle : a-t-on bien les 54 pays et 12 indicateurs ?
print('Pays trouvés      :', df['Country Code'].nunique(), '/ 54')
print('Indicateurs trouvés :', df['Indicator Code'].nunique(), '/ 12')
manquants = set(PAYS_AFRIQUE) - set(df['Country Code'])
print('Pays absents du fichier :', manquants if manquants else 'aucun')

Après filtrage : (648, 70)


Pays trouvés      : 54 / 54
Indicateurs trouvés : 12 / 12
Pays absents du fichier : aucun


---
## 4. Dépivotage (unpivot) des colonnes-années

On transforme les colonnes `1960, 1961, ... 2024` en deux colonnes `Year` / `Value` (format long).

In [5]:
id_cols = ['Country Code', 'Indicator Code']
annees = [c for c in df.columns if c.isdigit()]

long = df.melt(id_vars=id_cols, value_vars=annees, var_name='Year', value_name='Value')
long['Year'] = long['Year'].astype(int)

# On restreint à la période d'analyse 2000-2023
long = long[(long['Year'] >= ANNEE_MIN) & (long['Year'] <= ANNEE_MAX)]
print('Format long :', long.shape)
long.head(3)

Format long : (15552, 4)


,Country Code,Indicator Code,Year,Value
25920,DZA,EG.ELC.ACCS.ZS,2000,98.600000
25921,DZA,SP.DYN.TFRT.IN,2000,2.590000
25922,DZA,NY.GDP.PCAP.CD,2000,1772.928691


---
## 5. Repivotage : les 12 indicateurs deviennent des colonnes

Objectif : **une ligne = un pays × une année**, avec une colonne par indicateur. C'est le format idéal pour les cartes KPI et les nuages de points dans Power BI.

In [6]:
wide = long.pivot_table(index=['Country Code', 'Year'],
                        columns='Indicator Code',
                        values='Value',
                        aggfunc='first').reset_index()

# Renommer les codes en noms lisibles
wide = wide.rename(columns=INDICATEURS)
wide.columns.name = None
print('Format final (avant enrichissement) :', wide.shape)
wide.head(3)

Format final (avant enrichissement) : (1296, 14)


,Country Code,Year,ElectricityAccess,InternetUsers,GDPperCapita,AdultLiteracy,PrimaryEnrollment,U5Mortality,WaterAccess,SanitationAccess,MaternalMortality,LifeExpectancy,FertilityRate,Population
0,AGO,2000,24.2,0.105046,563.733796,NaN,NaN,185.0,37.842237,33.800408,659.0,46.501,6.639,16194869.0
1,AGO,2001,20.0,0.136014,533.586202,67.410004,NaN,176.3,38.338816,34.255855,629.0,47.032,6.601,16747208.0
2,AGO,2002,26.3,0.270377,999.065856,NaN,NaN,166.4,38.835392,34.711300,602.0,47.874,6.567,17327699.0


---
## 6. Analyse des valeurs manquantes

Étape clé d'un projet sérieux : on **mesure** les manques avant de décider quoi en faire.

In [7]:
ind_cols = list(INDICATEURS.values())
n = len(wide)
rapport = pd.DataFrame({
    'Renseignées': wide[ind_cols].notna().sum(),
    'Manquantes': wide[ind_cols].isna().sum(),
    '% rempli': (100 * wide[ind_cols].notna().sum() / n).round(0)
}).sort_values('% rempli')
rapport

,Renseignées,Manquantes,% rempli
AdultLiteracy,240,1056,19.0
PrimaryEnrollment,979,317,76.0
InternetUsers,1246,50,96.0
WaterAccess,1262,34,97.0
SanitationAccess,1269,27,98.0
GDPperCapita,1268,28,98.0
ElectricityAccess,1280,16,99.0
FertilityRate,1296,0,100.0
LifeExpectancy,1296,0,100.0
U5Mortality,1296,0,100.0


**Décision de traitement :**

| Situation | Indicateurs | Traitement |
|---|---|---|
| Bien remplis (96-100%) | espérance de vie, mortalités, fécondité, accès élec/eau/assainissement, PIB/hab, population, Internet | **Interpolation linéaire** dans chaque pays pour combler les rares trous (ces indicateurs évoluent lentement d'une année à l'autre → interpolation défendable). |
| Partiel (~76%) | scolarisation primaire | Même interpolation par pays. |
| Très lacunaire (~19%) | alphabétisation des adultes | Donnée d'enquête (recensements espacés). On la **conserve telle quelle** (pas d'invention de données sur de longues périodes) : Power BI l'affichera uniquement là où elle existe, en *instantané*, pas en tendance annuelle. |

---
## 7. Traitement des valeurs manquantes

In [8]:
wide = wide.sort_values(['Country Code', 'Year']).reset_index(drop=True)

# Indicateurs à interpoler (tous sauf l'alphabétisation, trop lacunaire)
a_interpoler = [c for c in ind_cols if c != 'AdultLiteracy']

def combler(groupe):
    # interpolation linéaire des trous intérieurs, puis remplissage des bords (ffill/bfill)
    return groupe.interpolate(method='linear', limit_direction='both')

wide[a_interpoler] = (wide.groupby('Country Code')[a_interpoler]
                          .transform(combler))

print('Manques restants après traitement :')
print(wide[ind_cols].isna().sum().sort_values(ascending=False))
print('\n(AdultLiteracy conserve ses manques : c\'est voulu et documenté.)')

Manques restants après traitement :
AdultLiteracy        1056
PrimaryEnrollment      24
LifeExpectancy          0
U5Mortality             0
FertilityRate           0
MaternalMortality       0
ElectricityAccess       0
WaterAccess             0
SanitationAccess        0
GDPperCapita            0
Population              0
InternetUsers           0
dtype: int64

(AdultLiteracy conserve ses manques : c'est voulu et documenté.)


---
## 8. Enrichissement : pays, région, groupe de revenu, sous-région

On rattache le dictionnaire des pays (`WDICountry.csv`) pour ajouter le nom complet, la région et le groupe de revenu, puis notre mapping de sous-région africaine.

In [9]:
pays = pd.read_csv(DATA / 'WDICountry.csv')
pays = pays[['Country Code', 'Short Name', 'Region', 'Income Group']].rename(
    columns={'Short Name': 'Country', 'Income Group': 'IncomeGroup'})

wide = wide.merge(pays, on='Country Code', how='left')
wide['SubRegion'] = wide['Country Code'].map(SOUS_REGION)

# Traduction lisible du groupe de revenu
revenu_fr = {
    'Low income': 'Revenu faible',
    'Lower middle income': 'Revenu intermédiaire (tranche inf.)',
    'Upper middle income': 'Revenu intermédiaire (tranche sup.)',
    'High income': 'Revenu élevé',
}
wide['IncomeGroupFR'] = wide['IncomeGroup'].map(revenu_fr)
wide[['Country Code', 'Country', 'Region', 'SubRegion', 'IncomeGroup']].drop_duplicates().head()

,Country Code,Country,Region,SubRegion,IncomeGroup
0,AGO,Angola,Sub-Saharan Africa,Afrique centrale,Lower middle income
24,BDI,Burundi,Sub-Saharan Africa,Afrique de l'Est,Low income
48,BEN,Benin,Sub-Saharan Africa,Afrique de l'Ouest,Lower middle income
72,BFA,Burkina Faso,Sub-Saharan Africa,Afrique de l'Ouest,Low income
96,BWA,Botswana,Sub-Saharan Africa,Afrique australe,Upper middle income


---
## 9. Colonnes calculées & mise en forme finale

In [10]:
# Arrondis cohérents par type d'indicateur
wide['Population']        = wide['Population'].round(0)
wide['GDPperCapita']      = wide['GDPperCapita'].round(0)
wide['FertilityRate']     = wide['FertilityRate'].round(2)
wide['MaternalMortality'] = wide['MaternalMortality'].round(0)
for c in ['LifeExpectancy','U5Mortality','PrimaryEnrollment','AdultLiteracy',
          'ElectricityAccess','WaterAccess','SanitationAccess','InternetUsers']:
    wide[c] = wide[c].round(1)

# Colonne calculée : PIB par habitant en milliers de $ (lisibilité)
wide['GDPperCapita_k'] = (wide['GDPperCapita'] / 1000).round(2)

# Ordre des colonnes
cols_ordre = ['Country Code', 'Country', 'Region', 'SubRegion', 'IncomeGroup', 'IncomeGroupFR', 'Year'] + ind_cols + ['GDPperCapita_k']
wide = wide[cols_ordre].sort_values(['Country', 'Year']).reset_index(drop=True)
wide.head()

,Country Code,Country,Region,SubRegion,IncomeGroup,IncomeGroupFR,Year,LifeExpectancy,U5Mortality,MaternalMortality,FertilityRate,PrimaryEnrollment,AdultLiteracy,ElectricityAccess,WaterAccess,SanitationAccess,GDPperCapita,Population,InternetUsers,GDPperCapita_k
0,DZA,Algeria,Middle East & North Africa,Afrique du Nord,Upper middle income,Revenu intermédiaire (tranche sup.),2000,70.6,41.7,123.0,2.59,109.7,NaN,98.6,89.0,84.5,1773.0,30903893.0,0.5,1.77
1,DZA,Algeria,Middle East & North Africa,Afrique du Nord,Upper middle income,Revenu intermédiaire (tranche sup.),2001,71.0,40.2,113.0,2.52,109.0,NaN,98.6,89.2,84.8,1896.0,31331221.0,0.6,1.90
2,DZA,Algeria,Middle East & North Africa,Afrique du Nord,Upper middle income,Revenu intermédiaire (tranche sup.),2002,71.6,38.6,102.0,2.43,110.7,69.9,98.6,89.3,85.0,1937.0,31750835.0,1.6,1.94
3,DZA,Algeria,Middle East & North Africa,Afrique du Nord,Upper middle income,Revenu intermédiaire (tranche sup.),2003,71.9,37.0,101.0,2.45,111.4,NaN,98.6,89.5,85.3,2284.0,32175818.0,2.2,2.28
4,DZA,Algeria,Middle East & North Africa,Afrique du Nord,Upper middle income,Revenu intermédiaire (tranche sup.),2004,72.5,35.4,91.0,2.50,111.7,NaN,98.6,89.7,85.6,2817.0,32628286.0,4.6,2.82


---
## 10. Contrôles qualité finaux

In [11]:
# 1) Pas de doublon pays-année
dup = wide.duplicated(subset=['Country Code', 'Year']).sum()
print('Doublons (pays, année) :', dup)

# 2) Dimensions attendues
print('Dimensions finales :', wide.shape, '| pays :', wide['Country'].nunique(), '| années :', wide['Year'].nunique())

# 3) Cohérence des plages de valeurs (sanity checks)
print('\nEspérance de vie  min/max :', wide['LifeExpectancy'].min(), '/', wide['LifeExpectancy'].max())
print('Accès électricité min/max :', wide['ElectricityAccess'].min(), '/', wide['ElectricityAccess'].max())
print('PIB/hab           min/max :', wide['GDPperCapita'].min(), '/', wide['GDPperCapita'].max())

# 4) Aperçu statistique
wide[ind_cols].describe().round(1)

Doublons (pays, année) : 0
Dimensions finales : (1296, 20) | pays : 54 | années : 24

Espérance de vie  min/max : 14.7 / 77.2
Accès électricité min/max : 0.8 / 100.0
PIB/hab           min/max : 110.0 / 19142.0


,LifeExpectancy,U5Mortality,MaternalMortality,FertilityRate,PrimaryEnrollment,AdultLiteracy,ElectricityAccess,WaterAccess,SanitationAccess,GDPperCapita,Population,InternetUsers
count,1296.0,1296.0,1296.0,1296.0,1272.0,240.0,1296.0,1296.0,1296.0,1296.0,1296.0,1296.0
mean,60.2,82.1,438.2,4.6,97.5,63.5,44.9,65.4,39.0,2258.0,20889563.4,15.7
std,7.7,48.2,317.5,1.4,23.3,19.7,29.7,18.1,26.2,2929.9,30899704.8,19.6
min,14.7,10.4,17.0,1.3,20.9,14.4,0.8,18.8,3.0,110.0,81131.0,0.0
25%,55.2,49.0,200.5,3.6,84.5,49.0,18.2,51.8,17.4,579.5,2439722.2,1.5
50%,60.1,74.8,396.5,4.7,101.1,66.4,42.0,64.6,33.6,1065.0,11235272.5,6.9
75%,64.9,111.8,584.2,5.6,111.8,79.0,65.2,79.0,53.7,2646.8,24909900.8,22.5
max,77.2,489.3,1662.0,7.8,163.2,97.3,100.0,99.9,99.7,19142.0,227882945.0,91.0


---
## 11. Export du jeu de données propre pour Power BI

In [12]:
sortie = OUT / 'Afrique_clean.csv'
wide.to_csv(sortie, index=False, encoding='utf-8-sig')
print('✅ Fichier écrit :', sortie.resolve())
print('   Lignes :', len(wide), '| Colonnes :', wide.shape[1])
print('\nColonnes :', list(wide.columns))

✅ Fichier écrit :

 C:\Projet DATA IA\ANALYSTLAB AFRICA\WEEK 8\Afrique_clean.csv
   Lignes : 1296 | Colonnes : 20

Colonnes : ['Country Code', 'Country', 'Region', 'SubRegion', 'IncomeGroup', 'IncomeGroupFR', 'Year', 'LifeExpectancy', 'U5Mortality', 'MaternalMortality', 'FertilityRate', 'PrimaryEnrollment', 'AdultLiteracy', 'ElectricityAccess', 'WaterAccess', 'SanitationAccess', 'GDPperCapita', 'Population', 'InternetUsers', 'GDPperCapita_k']


---
### Synthèse du nettoyage (à reprendre dans le rapport)

1. **Filtrage** : 397 000 lignes mondiales → 54 pays africains × 12 indicateurs.
2. **Restructuration** : dépivotage des années puis repivotage des indicateurs → 1 ligne par *pays × année*.
3. **Période** : recentrage sur **2000-2023**.
4. **Valeurs manquantes** : interpolation linéaire par pays pour les indicateurs bien remplis ; alphabétisation conservée telle quelle (donnée d'enquête lacunaire, traitée en instantané).
5. **Enrichissement** : nom du pays, région, groupe de revenu (+ version FR), **sous-région africaine**.
6. **Mise en forme** : arrondis cohérents, colonne calculée PIB/hab en milliers de $.
7. **Contrôles** : aucun doublon, plages de valeurs cohérentes.

➡️ Livrable : **`Afrique_clean.csv`**, prêt à être importé dans Power BI.